# Thực nghiệm bổ sung cho bài báo SpaceTime-DSCA (theo phiếu nhận xét DG25 – bản nộp lần 2)

Notebook này chạy toàn bộ thực nghiệm mà hội đồng yêu cầu, rồi **tự điền số liệu thật** vào bài báo v3
(`experiments/paper/PAPER_SPACETIME_DSCA_v3_template.docx`). Không có con số nào được gõ tay: mọi ô tô vàng trong bài
chỉ được thay bằng kết quả do các ô dưới đây tạo ra.

### Lộ trình

| Phần | Nội dung | Cần train? | Đáp ứng yêu cầu của hội đồng |
|---|---|---|---|
| **0** | Chuẩn bị: Drive, cập nhật mã, dữ liệu, kiểm tra checkpoint cũ | Không | – |
| **A** | Đánh giá lại checkpoint seed 42 trên **tập test** theo giao thức MFGF, chọn trọng số trên validation, bootstrap, đo tài nguyên | **Không** | 1, 2, một phần 3, Tiêu chí 3 |
| **B** | Lặp thêm 2 seed (1234, 2025) cho 3 tập con | **Có – 6 lần** | 3 |
| **C** | Ablation độc lập trên PRID (bỏ từng thành phần) | **Có – 4 lần** | 4 |
| **D** | Ablation cộng dồn trên PRID (đo lại Bảng 6) | Có – 3 lần (khuyến nghị) | 4 |
| **E** | Tổng hợp, vẽ hình, điền bài, tải về | Không | – |

**Nên làm theo thứ tự A → B → C → D → E.** Phần A cho ngay kết quả quan trọng nhất (so sánh cùng giao thức với MFGF).
Phần B, C, D có thể chia ra nhiều phiên Colab khác nhau.

### Quy tắc khi chạy
- Chọn **Runtime → Change runtime type → GPU** (T4 16 GB như các lần train trước, hoặc mạnh hơn).
- **Mỗi lần mở lại Colab** (mất kết nối, hết phiên): chạy lại các ô **0.2 → 0.5**, rồi chạy tiếp ô đang dở.
  Mọi bước đều *chạy lại được*: phần đã xong được bỏ qua, lần train đang dở được tiếp tục từ epoch gần nhất.
- Muốn xem trước lệnh sẽ chạy mà chưa chạy thật: thêm `"--dry_run"` vào lời gọi `rr(...)`.
- Kết quả nằm trong `srccv/results/` trên Drive (không mất khi Colab ngắt).

---
## Phần 0 – Chuẩn bị

### 0.1 Việc cần làm trên máy tính (trước khi mở notebook)

1. Tải file **`revision_code.zip`** (nằm trong thư mục `pblcv` trên máy) lên Google Drive, vào thư mục
   `MyDrive/pbl6remake/`. Gói này chứa: thư mục `experiments/` (script thực nghiệm + mẫu bài báo v3), `train.py` và
   `models/spacetime_vit.py`, `models/damr_main.py` đã thêm các cờ cho seed/ablation.
   *(Nếu quên, ô 0.4 sẽ hiện nút để tải lên trực tiếp.)*
2. Kiểm tra Drive có cấu trúc như lần train trước:

```
MyDrive/pbl6remake/
├── TVPReid_dataset.zip          ← dữ liệu (hoặc dataset.zip / thư mục TVPReid_dataset)
├── revision_code.zip            ← MỚI: tải lên ở bước 1
└── srccv/                       ← mã nguồn (train.py, models/, data/, utils/, configs/ ...)
    └── checkpoints/
        ├── TVPReid-PRID/best.pth    ← seed 42 của lần nộp trước (dùng lại, không train lại)
        ├── TVPReid-iLIDs/best.pth
        └── TVPReid-Duke/best.pth
```

3. Mở notebook này trên Colab (**File → Upload notebook**), chọn GPU, rồi chạy từng ô từ trên xuống.

### 0.2 Cấu hình đường dẫn
Chỉ sửa ô này nếu thư mục trên Drive của bạn khác với sơ đồ ở trên.

In [ ]:
# ===== 0.2 CẤU HÌNH (sửa nếu thư mục của bạn khác) =====
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/pbl6remake"      # thư mục dự án trên Drive
SRC_DIR   = DRIVE_PROJECT_ROOT + "/srccv"                     # mã nguồn (chứa train.py)
CODE_ZIP  = DRIVE_PROJECT_ROOT + "/revision_code.zip"         # gói mã bổ sung tải lên từ máy
DATA_ROOT = "/content/dataset"                                # dữ liệu giải nén ra ổ SSD của Colab (nhanh hơn Drive)
DATASET_SOURCES = ["TVPReid_dataset.zip", "dataset.zip", "TVPReid_dataset"]   # tìm lần lượt trong DRIVE_PROJECT_ROOT

SUBSETS   = ["TVPReid-PRID", "TVPReid-iLIDs", "TVPReid-Duke"]
SEEDS_NEW = [1234, 2025]                                      # Phần B: seed mới (seed 42 dùng lại checkpoint cũ)
LOO_RUNS  = ["no_init", "no_temporal", "no_dsca", "no_mim"]   # Phần C: ablation độc lập (bắt buộc)
CUM_RUNS  = ["cum1", "cum2", "cum3"]                          # Phần D: ablation cộng dồn (khuyến nghị)
print("OK – cấu hình đã nạp")

### 0.3 Kết nối Drive, kiểm tra GPU, cài thư viện
Colab sẽ hỏi quyền truy cập Google Drive – chọn tài khoản chứa thư mục `pbl6remake`.

In [ ]:
import os, sys, json, time, shutil, zipfile, codecs, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def run(cmd, check=True):
    # Chạy một lệnh shell trong SRC_DIR và in kết quả ngay khi có (kể cả thanh tiến trình của train.py).
    print("$", cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    p = subprocess.Popen(cmd, shell=True, cwd=SRC_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    dec = codecs.getincrementaldecoder("utf-8")("replace")
    while True:
        chunk = p.stdout.read1(65536)
        if not chunk:
            break
        sys.stdout.write(dec.decode(chunk))
        sys.stdout.flush()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh thất bại (mã {rc}) – đọc thông báo lỗi ở trên, xem Phụ lục 'Xử lý sự cố'.")
    return rc

def rr(stage, *extra):
    # Gọi experiments/run_revision.py cho một bước (stage). Thêm "--dry_run" để chỉ in lệnh.
    return run(f"python experiments/run_revision.py --stage {stage} --data_root {DATA_ROOT} " + " ".join(extra))

def seeds_to_train(sub):
    # Seed mới của Phần B; thêm seed 42 nếu không còn checkpoint của lần nộp trước cho tập con này.
    has_42 = Path(SRC_DIR, "checkpoints", sub, "best.pth").exists() or \
             Path(SRC_DIR, "checkpoints", sub, "seed42", "best.pth").exists()
    return "--seeds " + " ".join(map(str, SEEDS_NEW + ([] if has_42 else [42])))

import torch
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {prop.name}, {prop.total_memory / 1024**3:.1f} GB")
    if prop.total_memory < 14 * 1024**3:
        print("⚠️ GPU dưới 14 GB: train với lô 16 có thể hết bộ nhớ – nên đổi sang T4/L4/A100.")
else:
    print("⚠️ CHƯA CÓ GPU: Runtime → Change runtime type → GPU. Phần A.4 và mọi bước train đều cần GPU.")

if not os.path.isdir(SRC_DIR):
    raise FileNotFoundError(f"Không thấy {SRC_DIR}. Kiểm tra lại ô 0.2 hoặc Drive đã kết nối chưa.")
if IN_COLAB:
    run("pip install -q transformers pyyaml sentencepiece sacremoses")
print("OK")

### 0.4 Cập nhật mã nguồn trên Drive
Ô này giải nén `revision_code.zip` vào `srccv/`. Các file cũ bị ghi đè (`train.py`, hai file trong `models/`) được sao lưu
vào `srccv/_backup_before_revision/<thời điểm>/`. Khi gọi **không kèm cờ mới**, `train.py` vẫn chạy y như trước.

In [ ]:
if not os.path.exists(CODE_ZIP):
    if IN_COLAB:
        from google.colab import files
        print(f"Chưa thấy {CODE_ZIP} → chọn file revision_code.zip trên máy để tải lên:")
        uploaded = files.upload()
        shutil.move(next(iter(uploaded)), CODE_ZIP)
    else:
        raise FileNotFoundError(CODE_ZIP)

stamp = time.strftime("%Y%m%d_%H%M%S")
backup = Path(SRC_DIR) / "_backup_before_revision" / stamp
with zipfile.ZipFile(CODE_ZIP) as z:
    names = [n for n in z.namelist() if not n.endswith("/")]
    for n in names:
        old = Path(SRC_DIR) / n
        if old.exists():
            (backup / n).parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(old, backup / n)
    z.extractall(SRC_DIR)
for sub in ("experiments", "models", "data", "utils", "loss"):
    for d in Path(SRC_DIR, sub).rglob("__pycache__"):
        shutil.rmtree(d, ignore_errors=True)
print(f"Đã cập nhật {len(names)} file" + (f"; bản cũ lưu ở {backup}" if backup.exists() else ""))

checks = {
    "train.py có cờ --seed/--tag/--no_temporal_attn":
        "--no_temporal_attn" in Path(SRC_DIR, "train.py").read_text(encoding="utf-8"),
    "models/spacetime_vit.py có use_temporal_attn":
        "use_temporal_attn" in Path(SRC_DIR, "models", "spacetime_vit.py").read_text(encoding="utf-8"),
    "experiments/run_revision.py": Path(SRC_DIR, "experiments", "run_revision.py").exists(),
    "mẫu bài báo v3": Path(SRC_DIR, "experiments", "paper", "PAPER_SPACETIME_DSCA_v3_template.docx").exists(),
}
for k, v in checks.items():
    print("✅" if v else "❌", k)
assert all(checks.values()), "Thiếu file – kiểm tra lại revision_code.zip"

### 0.5 Chuẩn bị dữ liệu TVPReid
Giải nén dữ liệu từ Drive ra `/content/dataset` (vài phút, chỉ lần đầu của mỗi phiên Colab), rồi kiểm tra mỗi tập con
có đủ `train.csv`, `val.csv`, `test.csv`. Số dòng của các file này nên khớp Bảng 2 bài MFGF: PRID 921/71/142,
iLIDs 488/37/75, Duke 3920/302/603 video.

In [ ]:
import pandas as pd

def caption_dirs(root):
    return sorted(p.parent for p in Path(root).rglob("captions") if p.is_dir()) if os.path.isdir(root) else []

os.makedirs(DATA_ROOT, exist_ok=True)
if not caption_dirs(DATA_ROOT):
    for name in DATASET_SOURCES:
        src = os.path.join(DRIVE_PROJECT_ROOT, name)
        if name.endswith(".zip") and os.path.isfile(src):
            print(f"Giải nén {src} → {DATA_ROOT} ...")
            with zipfile.ZipFile(src) as z:
                z.extractall(DATA_ROOT)
            break
        if os.path.isdir(src):
            print(f"Sao chép {src} → {DATA_ROOT} ...")
            shutil.copytree(src, os.path.join(DATA_ROOT, name), dirs_exist_ok=True)
            break

EXPECTED_SPLIT = {"PRID": (921, 71, 142), "ILIDS": (488, 37, 75), "DUKE": (3920, 302, 603)}
found = caption_dirs(DATA_ROOT)
if not found:
    raise FileNotFoundError(f"Không tìm thấy thư mục 'captions/' nào trong {DATA_ROOT}. "
                            f"Kiểm tra tên file dữ liệu trong DATASET_SOURCES (ô 0.2).")
for d in found:
    counts = []
    for split in ("train", "val", "test"):
        f = d / "captions" / f"{split}.csv"
        counts.append(len(pd.read_csv(f)) if f.exists() else None)
    key = next((k for k in EXPECTED_SPLIT if k in d.name.upper().replace("-", "")), None)
    exp = EXPECTED_SPLIT.get(key)
    mark = "✅" if exp and tuple(counts) == exp else "⚠️"
    print(f"{mark} {d.name}: train/val/test = {counts}" + (f"  (Bảng 2 MFGF: {exp})" if exp else ""))

### 0.6 Kiểm tra checkpoint seed 42 của lần nộp trước
Phần A dùng lại `checkpoints/<tập con>/best.pth`. Ô này đọc epoch và Rank@1 validation lưu trong checkpoint để chắc chắn
đó đúng là checkpoint đã cho số liệu trong bài (PRID epoch 23 – 43,66%; iLIDs epoch 43 – 45,95%; Duke epoch 29 –
39,07%). Mỗi file ~2 GB nên ô này mất khoảng 1–3 phút.
- **❌ không có checkpoint**: tập con đó sẽ được train lại seed 42 ở Phần B (tự động).
- **⚠️ khác epoch/Rank@1**: vẫn dùng được, nhưng phải sửa câu "epoch được chọn ..." ở mục 5.2 và chú thích Bảng 5.

In [ ]:
EXPECTED_CKPT = {"TVPReid-PRID": (23, 43.66), "TVPReid-iLIDs": (43, 45.95), "TVPReid-Duke": (29, 39.07)}
missing = []
for sub in SUBSETS:
    p = Path(SRC_DIR, "checkpoints", sub, "best.pth")
    if not p.exists():
        others = sorted(x.name for x in p.parent.glob("*.pth")) if p.parent.exists() else []
        print(f"❌ {sub}: không có {p} (file .pth khác trong thư mục: {others or 'không có'})")
        missing.append(sub)
        continue
    ck = torch.load(p, map_location="cpu", weights_only=False)
    ep, r1 = ck.get("epoch"), (ck.get("metrics") or {}).get("R@1")
    seed = ((ck.get("config") or {}).get("train") or {}).get("seed")
    exp_ep, exp_r1 = EXPECTED_CKPT[sub]
    ok = ep == exp_ep and r1 is not None and abs(r1 - exp_r1) < 0.01
    r1_txt = f"{r1:.2f}" if r1 is not None else "?"
    print(f"{'✅' if ok else '⚠️'} {sub}: epoch {ep}, Rank@1 validation {r1_txt}%, seed {seed} "
          f"(trong bài: epoch {exp_ep}, {exp_r1}%) – {p.stat().st_size / 1024**3:.2f} GB")
    del ck
if missing:
    print("→ Các tập con sẽ train lại seed 42 ở Phần B (tự động):", missing)

try:
    st = os.statvfs("/content/drive/MyDrive" if IN_COLAB else SRC_DIR)
    print(f"\nDung lượng Drive còn trống: {st.f_bavail * st.f_frsize / 1024**3:.1f} GB "
          f"(mỗi lần train cần ~4 GB tạm thời; xong sẽ tự thu gọn còn ~0,7 GB hoặc 0 GB với ablation)")
except Exception as e:
    print("Không đọc được dung lượng Drive:", e)

### 0.7 (Tùy chọn) Giải phóng dung lượng Drive
Chỉ chạy nếu Drive gần đầy. Ô này **xoá** `checkpoints/<tập con>/latest.pth` của lần train trước (chỉ dùng để train
tiếp, không cần nữa) và bỏ trạng thái optimizer khỏi `best.pth` (giữ nguyên trọng số mô hình, epoch, chỉ số, cấu hình) –
tiết kiệm khoảng 4–5 GB mỗi tập con. Đổi `FREE_DRIVE_SPACE = True` rồi chạy.

In [ ]:
FREE_DRIVE_SPACE = False      # ← đổi thành True để thực hiện

if FREE_DRIVE_SPACE:
    freed = 0
    for sub in SUBSETS:
        d = Path(SRC_DIR, "checkpoints", sub)
        latest = d / "latest.pth"
        if latest.exists():
            freed += latest.stat().st_size
            latest.unlink()
            print("Đã xoá", latest)
        best = d / "best.pth"
        if best.exists():
            ck = torch.load(best, map_location="cpu", weights_only=False)
            if "optimizer_state_dict" in ck:
                before = best.stat().st_size
                keep = {k: ck[k] for k in ("model_state_dict", "epoch", "metrics", "config", "history") if k in ck}
                torch.save(keep, best)
                freed += before - best.stat().st_size
                print("Đã bỏ optimizer khỏi", best)
            del ck
    print(f"Giải phóng khoảng {freed / 1024**3:.1f} GB")
else:
    print("Bỏ qua (FREE_DRIVE_SPACE = False)")

---
## Phần A – Không cần train (chạy ngay)

Dùng checkpoint seed 42 sẵn có. Sau phần này đã có: so sánh cùng giao thức với MFGF (yêu cầu 1), trọng số chọn trên
validation (yêu cầu 2), khoảng tin cậy bootstrap (một phần yêu cầu 3), kiểm tra phân chia dữ liệu (Tiêu chí 3) và đo tài nguyên.
Yêu cầu 5 (demo, tiếng Việt) được xử lý bằng câu chữ trong bài: hệ thống chỉ dùng câu mô tả tiếng Anh.

### A.1 Kiểm tra phân chia train/val/test (Tiêu chí 3)
Đếm số video, số câu, số danh tính (person ID) của từng tập và số danh tính dùng chung giữa train và test.

Cách đọc kết quả:
- `match=True`: kích thước tập khớp Bảng 2 bài MFGF.
- `video_id overlaps` **phải bằng 0** ở cả ba cặp.
- `IDs shared ... train_test`: số danh tính xuất hiện cả ở train và test – con số này được ghi vào Bảng 3 của bài.
- `example ids`: person ID được lấy là **số đầu tiên** trong tên video (giống bộ nạp dữ liệu lúc train). Kiểm tra bằng
  mắt rằng số đó đúng là mã người; nếu không (ví dụ lấy nhầm số camera), dừng lại vì Bảng 3 sẽ sai.

In [ ]:
rr("split")

### A.2 Mã hóa tập validation và tập test bằng checkpoint seed 42
Mỗi video được mã hóa một lần; embedding lưu trong `results/emb/<tập con>/seed42/`. Khoảng vài phút cho cả ba tập con.

In [ ]:
rr("encode", "--seeds 42", "--no_ablations")

### A.3 Chọn trọng số trên validation, đánh giá tập test một lần (yêu cầu 1 và 2)
- Trọng số (w1, w2, w3) được chọn bằng tìm kiếm lưới bước 0,05 trên **validation**.
- Tập test được đánh giá theo đúng giao thức MFGF: mỗi video **một câu** truy vấn (câu thứ nhất), gallery 142/75/603
  video, khớp theo video. Kết quả với câu thứ hai và với giao thức hai câu/video cũng được lưu.

In [ ]:
rr("eval", "--seeds 42", "--no_ablations")

MFGF_R1 = {"TVPReid-PRID": 32.3, "TVPReid-iLIDs": 30.0, "TVPReid-Duke": 35.2}
print("\nKết quả seed 42 trên TẬP TEST, giao thức MFGF:")
for sub in SUBSETS:
    f = Path(SRC_DIR, "results", "runs", sub, "seed42.json")
    if not f.exists():
        print(f"  ⚠️ {sub}: chưa có kết quả")
        continue
    r = json.loads(f.read_text(encoding="utf-8"))
    sel = r["modes"]["val_selected"]
    t = sel["test"]["mfgf"]["metrics"]
    print(f"  {sub}: w={tuple(sel['weights'])} | R@1 {t['R@1']:.2f}  R@5 {t['R@5']:.2f}  R@10 {t['R@10']:.2f}  "
          f"MdR {t['MdR']:.1f}  ({r['n_test_videos']} video test) | MFGF R@1 {MFGF_R1[sub]}")

### A.4 Đo bộ nhớ GPU và độ trễ (thay cho con số "ít hơn 40%" chưa có bằng chứng)
So sánh bộ mã hóa SpaceTimeViT với tổ hợp ViT-B/16 (4 khung) + S3D (16 khung) của MFGF trên cùng GPU. **Bắt buộc có
GPU**; chạy khoảng 1–3 phút.

In [ ]:
rr("bench")
print(Path(SRC_DIR, "results", "bench.json").read_text(encoding="utf-8"))

### A.6 Tổng hợp tạm thời và xem bài đã điền được bao nhiêu
Ô này có thể chạy lại bất cứ lúc nào. Các ô của seed 1234/2025 và ablation vẫn còn tô vàng cho tới khi xong Phần B–D.

In [ ]:
from IPython.display import Markdown, display
rr("aggregate")
rr("figures")
run("python experiments/fill_paper.py", check=False)     # mã thoát 2 = còn token chưa có số, là bình thường lúc này
display(Markdown(Path(SRC_DIR, "results", "tables.md").read_text(encoding="utf-8")))

---
## Phần B – Lặp seed (bắt buộc, yêu cầu 3)

Hội đồng yêu cầu **ít nhất 3 seed cho cả ba tập con**. Seed 42 đã có, cần train thêm seed 1234 và 2025:
**6 lần train**, cùng cấu hình như lần trước (lr 1e-4, lô 16; PRID 30 epoch, iLIDs 50 epoch, Duke 30 epoch).

- Mỗi ô dưới đây train một tập con. Thời gian mỗi lần train xấp xỉ thời gian các lần train trước của nhóm;
  Duke lâu nhất (3.920 video train) nên để cuối, có thể cần 2 phiên Colab.
- Log in `Epoch xx/yy Complete | ... Elapsed: ...s` sau mỗi epoch để ước lượng thời gian còn lại.
- **Mất kết nối giữa chừng?** Chạy lại 0.2 → 0.5 rồi chạy lại đúng ô đó: lần train dở được tiếp tục từ epoch gần nhất.
- Train xong mỗi lần, script tự mã hóa val/test rồi thu gọn checkpoint (xoá `latest.pth`, bỏ optimizer) để Drive
  không bị đầy.

In [ ]:
# B.1 – TVPReid-PRID: seed 1234 và 2025
rr("train", "--subs TVPReid-PRID", seeds_to_train("TVPReid-PRID"), "--no_ablations")

In [ ]:
# B.2 – TVPReid-iLIDs: seed 1234 và 2025
rr("train", "--subs TVPReid-iLIDs", seeds_to_train("TVPReid-iLIDs"), "--no_ablations")

In [ ]:
# B.3 – TVPReid-Duke: seed 1234 và 2025 (lâu nhất)
rr("train", "--subs TVPReid-Duke", seeds_to_train("TVPReid-Duke"), "--no_ablations")

### B.4 Đánh giá các seed mới
Chạy sau khi xong B.1–B.3 (chạy sớm hơn cũng được – lần nào chưa train xong sẽ được báo và bỏ qua).

In [ ]:
ALL_SEEDS = "--seeds 42 " + " ".join(map(str, SEEDS_NEW))
rr("encode", ALL_SEEDS, "--no_ablations")
rr("eval", ALL_SEEDS, "--no_ablations")
for sub in SUBSETS:
    vals = []
    for s in [42] + SEEDS_NEW:
        f = Path(SRC_DIR, "results", "runs", sub, f"seed{s}.json")
        if f.exists():
            m = json.loads(f.read_text(encoding="utf-8"))["modes"]["val_selected"]["test"]["mfgf"]["metrics"]
            vals.append(f"seed {s}: {m['R@1']:.2f}")
        else:
            vals.append(f"seed {s}: chưa có")
    print(f"{sub} – R@1 test: " + " | ".join(vals))

---
## Phần C – Ablation độc lập trên PRID (bắt buộc, yêu cầu 4)

Bỏ **từng** thành phần khỏi mô hình đầy đủ, giữ nguyên phần còn lại (Bảng 6b mới), seed 42:

| Lần train | Cờ trong `train.py` | Ý nghĩa |
|---|---|---|
| `no_init` | `--no_pretrained_vit` | không khởi tạo ViT từ ImageNet-1K |
| `no_temporal` | `--no_temporal_attn` | bỏ chú ý thời gian trong mọi khối |
| `no_dsca` | `--no_dsca` | không phân rã câu: T_app = T_mot = T_full |
| `no_mim` | `--lambda_mim 0` | bỏ ràng buộc trực giao ℒ_MIM |

Dòng "bỏ chấm điểm đa tầng" tính trực tiếp từ mô hình đầy đủ, không cần train. Tổng cộng **4 lần train PRID**.

In [ ]:
rr("train", "--subs TVPReid-PRID", "--seeds 42", "--ablation_runs " + " ".join(LOO_RUNS))

In [ ]:
# Đánh giá các lần ablation độc lập
ARGS_C = ("--subs TVPReid-PRID", "--seeds 42", "--ablation_runs " + " ".join(LOO_RUNS))
rr("encode", *ARGS_C)
rr("eval", *ARGS_C)

---
## Phần D – Ablation cộng dồn trên PRID (khuyến nghị)

Bảng 6 cũ không có log đi kèm, và dòng 5 của nó (40,14) mâu thuẫn với Bảng 7 (38,03). Ba lần train dưới đây đo lại
Bảng 6 trên tập test bằng đúng quy trình mới (dòng 4–6 dùng lại các lần train ở Phần B, C):

| Lần train | Cấu hình |
|---|---|
| `cum1` | baseline: ViT từ đầu, không chú ý thời gian, BERT toàn câu, chỉ nhánh toàn thể |
| `cum2` | + khởi tạo ImageNet-1K |
| `cum3` | + chú ý không–thời gian phân chia |

**Nếu bỏ qua phần này**, Bảng 6 trong bài sẽ còn ô vàng: chỉ được giữ số cũ (ghi rõ là đo trên validation) nếu nhóm
còn log của các lần chạy cũ.

In [ ]:
rr("train", "--subs TVPReid-PRID", "--seeds 42", "--ablation_runs " + " ".join(CUM_RUNS))
ARGS_D = ("--subs TVPReid-PRID", "--seeds 42", "--ablation_runs " + " ".join(CUM_RUNS))
rr("encode", *ARGS_D)
rr("eval", *ARGS_D)

---
## Phần E – Tổng hợp, điền bài, tải về

### E.1 Tổng hợp và vẽ hình
Tính trung bình ± độ lệch chuẩn qua seed, khoảng tin cậy bootstrap (10.000 lần lấy mẫu lại), chênh lệch ablation (bootstrap
cặp), sinh các câu nhận xét từ khoảng tin cậy, vẽ Hình 1, 3, 5.

In [ ]:
from IPython.display import Image, Markdown, display
rr("aggregate")
rr("figures")
display(Markdown(Path(SRC_DIR, "results", "tables.md").read_text(encoding="utf-8")))
for f in ("fig1_test_r1.png", "fig3_delta_mfgf.png", "fig5_ablation.png"):
    p = Path(SRC_DIR, "results", "figures", f)
    if p.exists():
        print(f)
        display(Image(filename=str(p), width=520))

### E.2 Điền số liệu vào bài báo
Tạo `results/PAPER_SPACETIME_DSCA_v3.docx` từ mẫu. Ô in ra danh sách token **chưa có số** (nếu có) – chúng vẫn tô vàng
trong bài. Khi mọi token đã có số, dòng ghi chú "BẢN NHÁP" ở đầu bài tự động bị xoá.

In [ ]:
rc = run("python experiments/fill_paper.py", check=False)
print("\n✅ Đã điền đủ mọi số liệu" if rc == 0 else "\n⚠️ Còn token chưa có số – xem danh sách ở trên (chạy nốt Phần B/C/D)")

### E.3 Lưu kết quả và tải bài về máy
Nén toàn bộ `results/` (log, thứ hạng từng truy vấn, embedding, bảng, hình) thành một file trên Drive để nộp kèm khi
hội đồng cần kiểm tra lại, rồi tải bài đã điền về máy.

In [ ]:
out_zip = shutil.make_archive(os.path.join(DRIVE_PROJECT_ROOT, "results_revision_" + time.strftime("%Y%m%d_%H%M")),
                              "zip", root_dir=SRC_DIR, base_dir="results")
print("Đã lưu:", out_zip)
paper = Path(SRC_DIR, "results", "PAPER_SPACETIME_DSCA_v3.docx")
if IN_COLAB and paper.exists():
    from google.colab import files
    files.download(str(paper))

### E.4 Việc còn lại sau khi tải bài về
1. Mở `PAPER_SPACETIME_DSCA_v3.docx` bằng Word, tìm xem còn chỗ tô vàng nào không.
2. **Đọc lại các câu được sinh tự động** (tóm tắt, mục 5.1, 5.3, 5.5 và kết luận). Chúng chỉ nói điều mà khoảng tin
   cậy cho phép; nếu muốn diễn đạt khác, chỉ sửa câu chữ, **không sửa số**.
3. Nếu ô 0.6 báo checkpoint khác epoch 23/43/29: sửa câu "epoch được chọn ..." ở mục 5.2 và chú thích Bảng 5.
4. Nếu bỏ qua Phần D: xử lý Bảng 6 như ghi chú ở Phần D.
5. Xuất PDF (File → Save as → PDF).
6. Bản giải trình `GIAI_TRINH_DG25_lan3_NHAP.docx`: xoá dòng "BẢN NHÁP" ở đầu, điền ngày, rồi nộp kèm bài và file
   `results_revision_*.zip`.

---
## Phụ lục – Xử lý sự cố

| Hiện tượng | Nguyên nhân / cách xử lý |
|---|---|
| `WARNING: no captions found ... DUMMY index` | Không thấy dữ liệu: chạy lại ô 0.5, kiểm tra `DATA_ROOT` và `DATASET_SOURCES`. **Không dùng kết quả của lần chạy đó** – xoá `results/emb` và `results/runs` tương ứng rồi chạy lại. |
| `Checkpoint does not match the model` | Mã trên Drive chưa được cập nhật: chạy lại ô 0.4. |
| `CUDA out of memory` khi train | Cần GPU ≥ 15 GB (T4 16 GB như lần trước). Không giảm `batch_size` – lô 16 quyết định số mẫu âm của InfoNCE, đổi sẽ làm kết quả không so sánh được với seed 42. |
| Drive đầy / lỗi ghi file | Chạy ô 0.7; mỗi lần train cần ~4 GB tạm thời. |
| Colab ngắt giữa lúc train | Chạy lại 0.2 → 0.5, rồi chạy lại đúng ô đang dở (tự tiếp tục từ epoch gần nhất). |
| Muốn chạy lại một bước đã xong | Thêm `"--force"` vào lời gọi `rr(...)` của bước encode/eval. |

Chi tiết từng script: `experiments/README.md`.